# ⚡ Electronics RAG Pipeline & MCP Server Walkthrough

This Jupyter notebook provides an interactive, step-by-step walkthrough of the **Electronics RAG (Retrieval-Augmented Generation) Pipeline** and **Model Context Protocol (MCP) Server** architecture.

### System Architecture Overview:
```mermaid
flowchart TD
    A[General Electronics Documents] --> B[Python Ingestion Pipeline]
    B --> C["Clean, Split & Classify Knowledge"]
    C --> D[Generate Embeddings]
    D --> E[(Vector Database)]
    
    F[User / Admin] --> G[MCP Connection UI]
    G --> H[Enter MCP Server Command / URL]
    H --> I[Configure Access & Settings]
    I --> J[Test Connection & List Tools]
    
    K[Other AI Agent] -->|1. Send Electronics Question| L[Python MCP Server]
    L <-->|2. Retrieve Chunks & Embeddings| E
    L -->|3. Return Relevant Knowledge & Sources| K
    K --> M[Generate Final Electronics Answer]
```

## 1. Setup & Environment Imports

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in Python path
PROJECT_ROOT = Path(os.path.abspath('..'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.infrastructure.config import settings
from src.infrastructure.loaders import UniversalDocumentLoader
from src.infrastructure.splitters import CleanElectronicsSplitter
from src.infrastructure.classifier import ElectronicsDomainClassifier
from src.infrastructure.embeddings import get_embedding_model
from src.infrastructure.vector_store import ChromaVectorStore
from src.application.ingestion import IngestDocumentsUseCase
from src.application.retrieval import RetrieveKnowledgeUseCase
from src.application.mcp_service import MCPService
from src.domain.entities import ClassificationCategory

print(f"✅ Environment Ready! Data Directory: {settings.DATA_DIR}")

## 2. Ingestion Pipeline: Load Raw Electronics Documents

In [ ]:
loader = UniversalDocumentLoader()
docs = loader.load(str(settings.DOCS_DIR))

print(f"Loaded {len(docs)} technical electronics documents:")
for d in docs:
    print(f" - 📄 {d.title} ({d.file_type.upper()}) | Length: {len(d.content)} chars")

## 3. Clean, Split & Classify Knowledge into Electronics Domains

In [ ]:
splitter = CleanElectronicsSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)

print(f"Generated {len(chunks)} classified chunks:\n")
for i, chunk in enumerate(chunks[:5]):
    print(f"[Chunk #{i+1}] Doc: '{chunk.source_title}' | Category: {chunk.category.value} | Confidence: {chunk.confidence}")
    print(f"Tags: {chunk.tags}")
    print(f"Preview: {chunk.clean_content[:120]}...")
    print("-" * 80)

## 4. End-to-End Ingestion Pipeline Execution

In [ ]:
ingester = IngestDocumentsUseCase()
result = ingester.execute()

print("Ingestion Pipeline Results:")
for k, v in result.items():
    print(f"  {k}: {v}")

## 5. Semantic Search & Knowledge Retrieval (Vector DB)

In [ ]:
retriever = RetrieveKnowledgeUseCase()

# Test Query 1: Buck Converter Sizing
query_1 = "How do I calculate the inductor for a buck converter and prevent saturation?"
res_1 = retriever.execute(query_text=query_1, top_k=2)

print(f"🔍 Query: {query_1}\n")
print(res_1["formatted_context_for_llm"])

In [ ]:
# Test Query 2: Microcontroller SPI Pins with Category Filtering
query_2 = "What are the VSPI and HSPI pinouts on ESP32?"
res_2 = retriever.execute(
    query_text=query_2,
    category=ClassificationCategory.MICROCONTROLLERS_EMBEDDED.value,
    top_k=2
)

print(f"🔍 Query: {query_2}\n")
print(res_2["formatted_context_for_llm"])

## 6. MCP Server Tools Inspection & Simulation

In [ ]:
mcp_service = MCPService()
tools = mcp_service.get_registered_tools()

print(f"Exposing {len(tools)} MCP Tools to AI Agents:")
for tool in tools:
    print(f"⚡ Tool: {tool.name}")
    print(f"   Description: {tool.description}")
    print(f"   Required Inputs: {tool.input_schema.get('required', [])}\n")

## 7. AI Agent Execution Simulation
Here we simulate the complete interaction where an external AI Agent receives a user prompt, calls the Electronics MCP Server tool to retrieve grounded citations, and synthesizes the final engineering answer.

In [ ]:
def simulate_ai_agent_workflow(user_electronics_question: str):
    print(f"🤖 [Other AI Agent] Received Question: '{user_electronics_question}'")
    print("🤖 [Other AI Agent] Calling MCP tool: 'query_electronics_knowledge'...")
    
    # 1. AI agent invokes MCP tool
    mcp_response = mcp_service.query_electronics(
        query=user_electronics_question,
        top_k=2
    )
    
    print(f"⚡ [MCP Server] Returned {mcp_response['results_count']} citations from Vector DB.\n")
    
    # 2. AI agent generates final answer grounded in retrieved knowledge
    print("🤖 [Other AI Agent] Synthesizing Final Answer with Citations:\n")
    print("=" * 80)
    print("### Final Electronics Answer:")
    print(f"Based on the retrieved technical documentation from '{mcp_response['citations'][0]['source_title']}':")
    print(mcp_response["formatted_context_for_llm"])
    print("=" * 80)

# Run simulation with op-amp inquiry
simulate_ai_agent_workflow("What is the difference between inverting and non-inverting op-amp configurations?")

## 8. Starting the MCP Setup UI

To launch the Admin MCP Setup UI (strictly for configuration and tool testing, matching the architecture design):
```bash
streamlit run src/presentation/ui/app.py
```